# Pixel Transformer tutorial: from raw hits to a classifier

This is the short tutorial notebook. It jumps straight into a small Transformer-like model for pixel clusters.

The goal is not to build the most complicated architecture. The goal is to understand the core idea:

> represent each hit, let hits exchange information through attention, pool the learned hit information, then classify the cluster.

You will start from a simple implementation and then have space to try easy improvements. The leaderboard at the end compares:

- a pretrained BDT reference,
- a pretrained/simple Transformer reference,
- any models you train in this notebook.

In [ ]:
import sys, os, json, math, time, random
from pathlib import Path

NOTEBOOK_DIR = '/global/cfs/cdirs/m5197/sferrar2/ML4FPS/Jupyter'
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

import h5py
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

from dataloader import load_pixel_data, CLUSTER_FEATURE_KEYS
from tutorial_utils import (
    Trainer,
    plot_score_distributions, plot_confusion_matrix,
    plot_roc_comparison, plot_leaderboard,
    evaluate_classifier, predict_model,
    count_parameters, display_model_card,
)

## Configuration

In [ ]:
SIGNAL_H5 = '../Samples_v2/signal.h5'
BIB_H5    = '../Samples_v2/BIB.h5'

MAX_RAW_HITS = 50
TEST_FRACTION = 0.2
VAL_FRACTION  = 0.1
RANDOM_SEED   = 42
BATCH_SIZE    = 256

OUTPUT_DIR = Path('simple_transformer_tutorial_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Put pretrained reference models here if you have them.
PRETRAINED_DIR = Path('../Pretrained')
PRETRAINED_BDT_PATH = PRETRAINED_DIR / 'bdt_gradient_boosting.joblib'
PRETRAINED_TRANSFORMER_PATH = PRETRAINED_DIR / 'simple_pixel_transformer.pt'

# If the pretrained files are not present, the notebook can train references from scratch.
TRAIN_REFERENCES_IF_MISSING = True

In [ ]:
def get_device():
    device = 'cpu'
    if torch.cuda.is_available():
        try:
            torch.zeros(1).cuda()
            device = 'cuda'
        except RuntimeError:
            print('CUDA device found but not usable. Falling back to CPU.')
    return device

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)
device = get_device()
print('Using device:', device)

## Load raw-hit data

In [ ]:
data = load_pixel_data(
    SIGNAL_H5, BIB_H5,
    max_hits=MAX_RAW_HITS,
    batch=BATCH_SIZE,
    test_frac=TEST_FRACTION,
    val_frac=VAL_FRACTION,
    seed=RANDOM_SEED,
)

train_loader = data['train']
val_loader   = data['val']
test_loader  = data['test']
labels       = np.asarray(data['labels']).astype(int)
idx_train    = np.asarray(data['idx_train'])
idx_val      = np.asarray(data['idx_val'])
idx_test     = np.asarray(data['idx_test'])

print(f"Loaded {len(labels):,} clusters: {labels.sum():,} signal, {(labels == 0).sum():,} BIB")
print(f"Train: {len(idx_train):,}, Val: {len(idx_val):,}, Test: {len(idx_test):,}")
print(f"Sequence length: {MAX_RAW_HITS}")
print('Input features per hit: energy, time, x, y')

## The simple Transformer

This is the intentionally compact implementation used in the tutorial.

A few simplifying choices are deliberate:

- one attention head,
- simple dot-product attention,
- no complicated masking inside the attention softmax,
- sum pooling at the end,
- small MLP classifier head.

The purpose is to make each operation easy to inspect and modify.

In [ ]:
class Attention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.q = nn.Linear(dim, dim)
        self.k = nn.Linear(dim, dim)
        self.v = nn.Linear(dim, dim)
        self.scale = dim ** -0.5

    def forward(self, x, mask):
        q = self.q(x)
        k = self.k(x)
        v = self.v(x)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        x = attn @ v

        return x * mask


class TransformerBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.att = Attention(dim)
        self.proj1 = nn.Linear(dim, dim)
        self.proj2 = nn.Linear(dim, dim)
        self.activation = nn.GELU()
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x, mask):
        x = x + self.att(self.norm1(x), mask)
        x = self.activation(self.proj1(x)) * mask
        x = x + self.proj2(self.norm2(x)) * mask
        return x


class SimplePixelTransformer(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=64, num_layers=3, num_classes=2):
        super().__init__()
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.blocks = nn.ModuleList([TransformerBlock(hidden_dim) for _ in range(num_layers)])
        self.output_layer = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        # A hit is real if the energy feature is nonzero.
        mask = (x[:, :, 0] != 0).unsqueeze(-1).float()

        x = self.input_layer(x) * mask
        for block in self.blocks:
            x = block(x, mask)

        # Sum pooling preserves occupancy information better than a pure mean.
        x = (x * mask).sum(dim=1) / (mask.shape[1] ** 0.5)
        return self.output_layer(x)

## Train or load the reference simple Transformer

In [ ]:
config = {
    'input_dim': 4,
    'hidden_dim': 64,
    'num_layers': 3,
    'num_classes': 2,
}

reference_transformer = SimplePixelTransformer(**config).to(device)
display_model_card(reference_transformer, config)

if PRETRAINED_TRANSFORMER_PATH.exists():
    print('Loading pretrained Transformer:', PRETRAINED_TRANSFORMER_PATH)
    reference_transformer.load_state_dict(torch.load(PRETRAINED_TRANSFORMER_PATH, map_location=device))
elif TRAIN_REFERENCES_IF_MISSING:
    print('No pretrained Transformer found. Training reference model...')
    trainer = Trainer(
        train_dataset=train_loader,
        val_dataset=val_loader,
        model=reference_transformer,
        lr=3e-4,
        optimizer=torch.optim.Adam,
        loss_fn=nn.CrossEntropyLoss,
        device=device,
    )
    trainer.train(num_epochs=60, patience=8)
    reference_transformer = trainer.model
    torch.save(reference_transformer.state_dict(), OUTPUT_DIR / 'simple_pixel_transformer_reference.pt')
else:
    print('Using randomly initialized reference_transformer. Set TRAIN_REFERENCES_IF_MISSING=True to train.')

## Optional student exercise: try one improvement

Ideas that keep the model simple:

- **Change the model capacity.** Increase `hidden_dim` from 64 to 96 or 128. This gives each hit more learned features, but it also increases training time.
- **Change the depth.** Try 2, 3, or 4 Transformer blocks. In our tests, 2--3 layers were usually enough; more layers did not automatically help.
- **Try a different pooling choice.** The reference uses sum pooling because it preserves some occupancy information. Mean pooling is also reasonable, but it divides out the number of hits.
- **Add light dropout.** Try `dropout=0.05` or `0.10` in the attention weights or classifier head. This can help if validation performance is noisy.
- **Preprocess hit `energy` and `time` before the input layer.** A useful simple version is:
  - `energy -> log1p(energy)`, because energy can have a long positive tail;
  - `time -> asinh(time)`, because time can have large positive values but may also have small/negative values;
  - then standardize each hit feature using the mean and standard deviation computed on real training hits only.

For example, conceptually:

```python
x[:, :, 0] = torch.log1p(torch.clamp(x[:, :, 0], min=0))
x[:, :, 1] = torch.asinh(x[:, :, 1])
x = (x - feature_mean) / feature_std
```

The important distinction is that this is still using primitive hit information. It is not giving the network engineered cluster quantities like `cluster_size_y` or `incident_angle`.

- **Give the model primitive detector context in the advanced notebook.** For example, cluster position variables like `(cluster_x, cluster_y, cluster_z, cluster_r)` helped because local pixel coordinates alone do not contain all detector-position information.

For the live tutorial, it is enough to modify the class above or copy it below into `MyPixelTransformer`.

In [ ]:
# Example student model: edit this cell or copy SimplePixelTransformer above.
class MyPixelTransformer(SimplePixelTransformer):
    pass

my_config = {
    'input_dim': 4,
    'hidden_dim': 64,
    'num_layers': 3,
    'num_classes': 2,
}

TRAIN_MY_MODEL = False

if TRAIN_MY_MODEL:
    my_model = MyPixelTransformer(**my_config).to(device)
    trainer = Trainer(
        train_dataset=train_loader,
        val_dataset=val_loader,
        model=my_model,
        lr=3e-4,
        optimizer=torch.optim.Adam,
        loss_fn=nn.CrossEntropyLoss,
        device=device,
    )
    trainer.train(num_epochs=60, patience=8)
    my_model = trainer.model
else:
    my_model = None

## Load the pretrained BDT reference for the leaderboard

The BDT is treated as a pretrained reference in this notebook. If the file is missing, run the BDT notebook first or update `PRETRAINED_BDT_PATH`.

In [ ]:
def as_feature_key_list(keys):
    if isinstance(keys, dict):
        keys = list(keys.keys())
    return [str(k) for k in keys]

FALLBACK_BDT_KEYS = [
    'cluster_energy', 'cluster_time', 'cluster_r', 'incident_angle',
    'cluster_size_tot', 'cluster_size_x', 'cluster_size_y',
    'cluster_rms_x', 'cluster_rms_y', 'cluster_skew_x', 'cluster_skew_y',
    'cluster_aspect', 'cluster_ecc',
]

try:
    BDT_FEATURE_KEYS = as_feature_key_list(CLUSTER_FEATURE_KEYS)
except Exception:
    BDT_FEATURE_KEYS = FALLBACK_BDT_KEYS


def load_cluster_feature_table(signal_h5, bib_h5, keys):
    tables = []
    for path in [signal_h5, bib_h5]:
        with h5py.File(path, 'r') as f:
            grp = f['clusters']
            arr = np.stack([grp[k][:] for k in keys], axis=1).astype('float32')
            tables.append(arr)
    return np.concatenate(tables, axis=0)

X_bdt_all = load_cluster_feature_table(SIGNAL_H5, BIB_H5, BDT_FEATURE_KEYS)
X_bdt_test = X_bdt_all[idx_test]
y_test = labels[idx_test]

if not PRETRAINED_BDT_PATH.exists():
    raise FileNotFoundError(
        f'Pretrained BDT not found at {PRETRAINED_BDT_PATH}. '
        'Run 01_bdt_baseline.ipynb first, or update PRETRAINED_BDT_PATH.'
    )

print('Loading pretrained BDT:', PRETRAINED_BDT_PATH)
bdt = joblib.load(PRETRAINED_BDT_PATH)


## Evaluation helpers and leaderboard

In [ ]:
def bib_rejection_at_signal_efficiency(y, scores, sig_eff):
    y = np.asarray(y).astype(int)
    scores = np.asarray(scores)
    threshold = np.quantile(scores[y == 1], 1.0 - sig_eff)
    bib_eff = np.mean(scores[y == 0] >= threshold)
    return 1.0 - bib_eff


def metrics_row(name, y, scores, kind='model'):
    return {
        'name': name,
        'kind': kind,
        'test_auc': roc_auc_score(y, scores),
        'test_bib_rej_at_sig_eff_0p80': bib_rejection_at_signal_efficiency(y, scores, 0.80),
        'test_bib_rej_at_sig_eff_0p90': bib_rejection_at_signal_efficiency(y, scores, 0.90),
        'test_bib_rej_at_sig_eff_0p95': bib_rejection_at_signal_efficiency(y, scores, 0.95),
    }


def score_torch_model(model, loader, name):
    scores, y = predict_model(model, loader, device)
    row = metrics_row(name, y, scores, kind='Transformer')
    return row, np.asarray(y), np.asarray(scores)

leaderboard_rows = []
score_payloads = {}

# BDT reference
if bdt is not None:
    bdt_scores = bdt.predict_proba(X_bdt_test)[:, 1]
    leaderboard_rows.append(metrics_row('BDT reference', y_test, bdt_scores, kind='BDT'))
    score_payloads['BDT reference'] = (y_test, bdt_scores)

# Simple Transformer reference
row, y_tf, scores_tf = score_torch_model(reference_transformer, test_loader, 'SimplePixelTransformer reference')
leaderboard_rows.append(row)
score_payloads['SimplePixelTransformer reference'] = (y_tf, scores_tf)

# Student model, if trained
if my_model is not None:
    row, y_my, scores_my = score_torch_model(my_model, test_loader, 'MyPixelTransformer')
    leaderboard_rows.append(row)
    score_payloads['MyPixelTransformer'] = (y_my, scores_my)

leaderboard = pd.DataFrame(leaderboard_rows).sort_values('test_auc', ascending=False)
display(leaderboard)
leaderboard.to_csv(OUTPUT_DIR / 'tutorial_leaderboard.csv', index=False)

In [ ]:
# Plots for the reference models
roc_payload = {}
for name, (y, scores) in score_payloads.items():
    try:
        roc_payload[name] = evaluate_classifier(y, scores, name)
    except Exception as e:
        print('Could not use evaluate_classifier for', name, e)

if roc_payload:
    plot_roc_comparison(roc_payload)

for name, (y, scores) in score_payloads.items():
    plot_score_distributions(y, scores, title=f'{name} score distribution')